1/x in encoder1 4670.44
1/x in encoder2 68336.18

In [1]:
import subprocess
from datasets import load_dataset
import re
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

dataset = load_dataset("dair-ai/emotion", split="test")

/Users/tonyma/code/FHE-BERT-Tiny-Emotion/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gokuls/BERT-tiny-emotion-intent")

texts = dataset['text']
token_length = [len(tokenizer.encode(entry['text'], truncation=False)) for entry in dataset]
labels = dataset['label']

df = pd.DataFrame({
    'text': texts,
    'token_length': token_length,
    'label': labels
})

# sort label
df_label = df.sort_values(by=['label', 'token_length'], ascending=[True, True]).reset_index(drop=True)

In [3]:
new_df2 = df_label.groupby('label').head(10).reset_index(drop=True)
print(len(new_df2))
print(new_df2[:21])

60
                            text  token_length  label
0          i feel so embarrassed             6      0
1             i do feel stressed             6      0
2                i feels so lame             6      0
3      i feel embarrassed enough             6      0
4          i stop feeling guilty             6      0
5         i feel depressed again             6      0
6              i feel soo lonely             6      0
7     im feeling depressed again             6      0
8           i just feel troubled             6      0
9              i feel less alone             6      0
10          im feeling energetic             5      1
11          i feel more creative             6      1
12        i feel really inspired             6      1
13         i feel more energetic             6      1
14             i feel any better             6      1
15       i would feel productive             6      1
16   im feeling hopeful relieved             6      1
17           i feel gorge

In [4]:
correct = []
error = []
output = []
execution_times = []

func_error = []
count = 0

def run_fhe_bert_tiny(sample):
    global correct, error, func_error, output, execution_times, count
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    cmd = [f"./FHE-BERT-Tiny", sentence]
    #cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct.append(count)
            print("sadness")
        else:
            error.append(count)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct.append(count)
            print("joy")
        else:
            error.append(count)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct.append(count)
            print("love")
        else:
            error.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct.append(count)
            print("anger")
        else:
            error.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct.append(count)
            print("fear")
        else:
            error.append(count)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct.append(count)
            print("surprise")
        else:
            error.append(count)
    else: # error
        func_error.append(count)
        
    print(seconds, "seconds")

In [5]:
from tqdm import tqdm

for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    run_fhe_bert_tiny(row)
    count+=1

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [04:08<4:04:23, 248.53s/it]

[7.44759, -1.49149, -2.57324, -0.815854, -1.26737, -2.6631]
sadness
237.0 seconds
sentence:  i do feel stressed


  3%|▎         | 2/60 [06:32<3:01:02, 187.28s/it]

...

[-2.04165, -0.550292, -0.0278767, -2.48904, 0.195694, 6.7721]
surprise
176.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [2:40:11<03:06, 186.18s/it]

[1.21647e+33, 1.22806e+34, -1.86586e+33, 5.20092e+33, 6.31517e+33, -2.89178e+33]
177.0 seconds
sentence:  i always feel very shocked by that me threatening


100%|██████████| 60/60 [2:43:31<00:00, 163.52s/it]

[-2.28401, 2.90605, -2.06354, -1.7037, 0.94312, 1.25579]
188.0 seconds


BELOW NEED CHECK & MODIFY

In [7]:
len(correct)

37

In [11]:
len(error)

23

In [6]:
error

[9,
 10,
 11,
 21,
 22,
 23,
 24,
 25,
 27,
 28,
 29,
 32,
 38,
 40,
 41,
 42,
 43,
 50,
 54,
 55,
 56,
 58,
 59]

wrong numbers
label 0 1/10
label 1 2/10
label 2 8/10 <-- why?
label 3 2/10
label 4 4/10
label 5 6/10

In [8]:
print(len(output))
print(len(execution_times))

60
60


In [9]:
import numpy as np
np.savetxt("output_labels_60", output, delimiter=",")
np.savetxt("execution_time_labels_60", output, delimiter=",")

Test again

In [10]:
import math

new_resamples = []
for i in range(len(output)):
    if (int(math.floor(math.log10(abs(output[i][0])))) > 0) or (int(math.floor(math.log10(abs(output[i][1])))) > 0) or (int(math.floor(math.log10(abs(output[i][2])))) > 0) or (int(math.floor(math.log10(abs(output[i][3])))) > 0) or (int(math.floor(math.log10(abs(output[i][4])))) > 0) or (int(math.floor(math.log10(abs(output[i][5])))) > 0):
        print(f"{i}: {output[i]}")
        new_resamples.append(i)
        
print(len(new_resamples))

9: [2.46359e+33, 7.30703e+33, -6.51302e+33, 7.57212e+32, 1.30694e+34, 3.09254e+33]
10: [-1.21765e+34, -3.42688e+32, 2.81306e+33, -9.2053e+33, 3.71391e+33, -1.52153e+34]
11: [1760.13, -386.664, -7274.23, -914.256, -5733.15, 4120.62]
24: [-3.23824e+33, 2.73933e+33, -1.11824e+34, 6.50081e+33, -6.12036e+33, -2.43276e+33]
27: [1.43154e+33, -2.17332e+33, 5.43872e+33, 3.3903e+33, 6.69834e+33, -1.26595e+34]
32: [7.88092e+31, -1.45827e+34, -4.11692e+33, 7.6396e+32, 1.18395e+34, 6.15561e+33]
38: [-2.10625e+33, -4.59286e+33, -5.8118e+33, -7.94676e+33, 1.14534e+34, -1.15184e+34]
40: [1.52873e+33, 8.14157e+33, -1.41791e+32, -8.73761e+33, 6.43341e+33, -7.45235e+31]
41: [-1.19939e+34, -8.9544e+32, -3.83811e+31, 1.11806e+34, 8.35269e+33, 1.1993e+34]
42: [-1.85857e+34, 7.99078e+33, 1.07358e+33, -1.52138e+34, 5.97526e+33, -4.72454e+33]
43: [9.86322e+33, -6.69033e+33, -3.31275e+33, -8.535e+33, -7.26989e+33, 7.97693e+33]
50: [13948.4, 2925.22, -272.525, 8505.86, 6859.01, -7372.15]
54: [-8.87593e+33, -8.47

In [12]:
new_resamples

[9, 10, 11, 24, 27, 32, 38, 40, 41, 42, 43, 50, 54, 56, 58]

In [13]:
for i in new_resamples:
    if not i in error:
        print("not exist: ", i)

In [18]:
correct2 = []
error2 = []
output2 = []
execution_times2 = []

func_error2 = []
count2 = 0

def run_fhe_bert_tiny2(sample):
    global correct2, error2, func_error2, output2, execution_times2, count2
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    #cmd = [f"./FHE-BERT-Tiny", sentence]
    cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output2.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times2.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct2.append(count2)
            print("sadness")
        else:
            error2.append(count2)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct2.append(count2)
            print("joy")
        else:
            error2.append(count2)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct2.append(count2)
            print("love")
        else:
            error2.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct2.append(count2)
            print("anger")
        else:
            error2.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct2.append(count2)
            print("fear")
        else:
            error2.append(count2)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct2.append(count2)
            print("surprise")
        else:
            error2.append(count2)
    else: # error
        error2.append(count2)
        
    print(seconds, "seconds")

In [19]:
from tqdm import tqdm

eliminated = 0
for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    if count2 == new_resamples[eliminated]:
        #print(count)
        eliminated+=1
        run_fhe_bert_tiny2(row)
    count2+=1
    
    if eliminated == len(new_resamples):
        break

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel less alone


 17%|█▋        | 10/60 [02:23<11:57, 14.34s/it]

[7.4451, -1.4135, -2.5332, -0.8929, -1.2567, -2.797]
sadness
132.0 seconds
sentence:  im feeling energetic


 18%|█▊        | 11/60 [04:38<24:05, 29.49s/it]

...

[3.6287, -1.1944, -3.0752, -0.6893, -0.5975, 1.7137]
167.0 seconds
sentence:  i replied feeling strange at giving the orders


 95%|█████████▌| 57/60 [37:49<02:49, 56.45s/it]

[-1.7634, -1.1617, -1.5179, -2.4162, 3.5534, 3.9431]
surprise
176.0 seconds
sentence:  i am feeling amazing and seeing the difference


 97%|█████████▋| 58/60 [40:57<01:24, 42.37s/it]

[-1.982, 3.2224, -1.2746, -2.7966, -1.1723, 3.7885]
surprise
176.0 seconds


In [20]:
len(correct2)

8

In [21]:
# total accuracy
total_accuracy = correct + correct2
total_accuracy.sort()
print(len(total_accuracy))

45


In [22]:
len(total_accuracy)/60

0.75

In [23]:
# Each label accuracy
label0_accuracy = []
label1_accuracy = []
label2_accuracy = []
label3_accuracy = []
label4_accuracy = []
label5_accuracy = []

for i in total_accuracy:
    if i < 10:
        label0_accuracy.append(i)
    elif i < 20:
        label1_accuracy.append(i)
    elif i < 30:
        label2_accuracy.append(i)
    elif i < 40:
        label3_accuracy.append(i)
    elif i < 50:
        label4_accuracy.append(i)
    elif i < 60:
        label5_accuracy.append(i)

In [24]:
print(len(label0_accuracy))
print(len(label1_accuracy))
print(len(label2_accuracy))
print(len(label3_accuracy))
print(len(label4_accuracy))
print(len(label5_accuracy))

10
9
2
9
8
7


In [38]:
print(df_label[df_label['label'] == 0].shape[0])
print(df_label[df_label['label'] == 1].shape[0])
print(df_label[df_label['label'] == 2].shape[0])
print(df_label[df_label['label'] == 3].shape[0])
print(df_label[df_label['label'] == 4].shape[0])
print(df_label[df_label['label'] == 5].shape[0])


581
695
159
275
224
66


In [29]:
new_resamples

[9, 10, 11, 24, 27, 32, 38, 40, 41, 42, 43, 50, 54, 56, 58]

In [30]:
new_resamples.index(11)

2

In [34]:
# check final wrong answer
final_wrong = []

for i in range(60):
    if not i in total_accuracy and i in new_resamples:
        print(i)
        print(f"first: {output[i]}")
        print(f"retest: {output2[new_resamples.index(i)]}")
        final_wrong.append(i)

11
first: [1760.13, -386.664, -7274.23, -914.256, -5733.15, 4120.62]
retest: [-1.6345667991798928e+34, 2.0814002811063276e+33, 6.102430064646848e+33, -9.76511671081185e+33, -1.1366409815534296e+34, 6.841857950291284e+33]
24
first: [-3.23824e+33, 2.73933e+33, -1.11824e+34, 6.50081e+33, -6.12036e+33, -2.43276e+33]
retest: [-1.9419, 6.422, 0.9851, -2.7518, -2.8562, -1.9908]
27
first: [1.43154e+33, -2.17332e+33, 5.43872e+33, 3.3903e+33, 6.69834e+33, -1.26595e+34]
retest: [1.460662031992953e+34, 1.452313545013633e+33, 1.1174845412853746e+34, -1.8380727012270171e+34, 7.389592414161397e+33, -1.8965434233433677e+32]
32
first: [7.88092e+31, -1.45827e+34, -4.11692e+33, 7.6396e+32, 1.18395e+34, 6.15561e+33]
retest: [1.2880301198814054e+34, 8.487844574510732e+33, -6.245700686320477e+33, -4.56028434003593e+33, -5.365614616622747e+32, 3.499349435926231e+33]
41
first: [-1.19939e+34, -8.9544e+32, -3.83811e+31, 1.11806e+34, 8.35269e+33, 1.1993e+34]
retest: [-7.974630747781933e+32, 1.7634581650150336e+3

In [26]:
print(len(final_wrong))

15
